In [9]:
import pandas as pd

In [10]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
caminho = "https://www-usr.inf.ufsm.br/~joaquim/UFSM/DM/ds/AgeOfMythology_work.csv"
df = pd.read_csv(caminho, dtype="string", keep_default_na=False, encoding="utf-8")

In [11]:
df

,Unit,Age,Cost,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,LOS,Train time,Bonus,Other,Class
0,Pegasus,1,50F 2Fv,2,--,--,--,0,140,50,50,99,5,18,6,,,Myth Unit
1,Centaur,2,200W 15Fv,3,--,12,--,12,220,25,35,80,5,20,19,"Myth units, -Heroes",,Myth Unit
2,Cyclops,2,250F 22Fv,4,15,--,12,2,600,40,50,80,3.2,16,20,Myth units,,Myth Unit
3,Minotaur,2,200F 16Fv,4,15,--,10,2,300,60,50,80,4,20,20,Myth units,,Myth Unit
4,Hydra,3,250F 28Fv,5,20,--,10,3,800,60,40,80,4,16,20,"Myth units, -Heroes",,Myth Unit
5,Manticore,3,300W 28Fv,5,--,11**,--,16,420,30,60,80,4.3,20,17,"Myth units, -Heroes",,Myth Unit
6,Nemean Lion,3,200G 25Fv,4,20,--,10,1,660,30,60,80,4.8,16,20,"Myth units, -Heroes",,Myth Unit
7,Colossus,4,300G 40Fv,5,20,--,50,3,1100_,50,80,80,2.4,16,24,"Myth units, -Heroes",,Myth Unit
8,Chimera,4,300G 30Fv,4,20,--,--,1,800,60,60,80,5.3,16,20,"Myth units, -Heroes",,Myth Unit
9,Medusa,4,250G 40Fv,5,--,15,12.5,10,360,60,70,80,4.3,18,20,"Myth units, -Heroes",,Myth Unit


In [12]:

import pandas as pd

RECURSOS = {
    "F": "cost_food",
    "W": "cost_wood",
    "G": "cost_gold",
    "Fv": "cost_favor",
}

COLUNAS_NUMERICAS = [
    "age", "pop", "dmg_hack", "dmg_pierce", "dmg_crush",
    "range", "hp", "armor_hack", "armor_pierce",
    "armor_crush", "speed", "los", "train_time",
]


def extrair_custos(valor):
    if pd.isna(valor) or valor in ("", "--"):
        return {nome: 0 for nome in RECURSOS.values()}

    custos = {nome: 0 for nome in RECURSOS.values()}

    for item in valor.replace(",", " ").split():
        recurso = next(
            (r for r in RECURSOS if item.endswith(r)),
            None
        )

        if recurso is None:
            raise ValueError(f"Custo inválido: {item}")

        quantidade = item.removesuffix(recurso)

        if not quantidade.isdigit():
            raise ValueError(f"Custo inválido: {item}")

        custos[RECURSOS[recurso]] += int(quantidade)

    return custos


def limpar_dataset(df) -> pd.DataFrame:
    df = df.copy()

    # Padroniza espaços e valores ausentes
    df.columns = [
        " ".join(col.replace("\u00a0", " ").split())
        for col in df.columns
    ]

    for coluna in df.columns:
        df[coluna] = df[coluna].map(
            lambda x: " ".join(x.replace("\u00a0", " ").split())
            if isinstance(x, str) else x
        )
    
    df = df.replace("", pd.NA)

    # Renomeia colunas
    df = df.rename(columns={
        "Unit": "unit",
        "Age": "age",
        "Cost": "cost_raw",
        "Pop": "pop",
        "DmgHack": "dmg_hack",
        "DmgPierce": "dmg_pierce",
        "DmgCrush": "dmg_crush",
        "Range": "range",
        "HP": "hp",
        "ArmorHack": "armor_hack",
        "ArmorPierce": "armor_pierce",
        "ArmorCrush": "armor_crush",
        "Speed": "speed",
        "LOS": "los",
        "Train time": "train_time",
        "Bonus": "bonus",
        "Other": "other",
        "Class": "unit_class",
    })

    # Corrige erros de escrita
    for coluna in ["unit_class", "bonus"]:
        df[coluna] = df[coluna].str.replace(
            "Infantary", "Infantry", regex=False
        )

    # Extrai custos
    custos = df["cost_raw"].apply(extrair_custos)
    custos = pd.DataFrame(custos.tolist(), index=df.index)

    for coluna in custos.columns:
        df[coluna] = custos[coluna].astype("Int64")

    # Remove marcações dos danos
    for coluna in ["dmg_hack", "dmg_pierce", "dmg_crush"]:
        df[coluna] = df[coluna].str.replace("*", "", regex=False)

    # Corrige HP
    df["hp"] = df["hp"].str.removesuffix("_")

    # Remove símbolo de porcentagem das armaduras
    for coluna in ["armor_hack", "armor_pierce", "armor_crush"]:
        df[coluna] = df[coluna].str.removesuffix("%")

    colunas_dmg = ["dmg_hack", "dmg_pierce", "dmg_crush"]

    df["does_attack"] = df[colunas_dmg].ne("--").any(axis=1).astype(int)


    # Converte colunas numéricas
    for coluna in COLUNAS_NUMERICAS:

        substituicoes = {
            "--": 0 if coluna.startswith("dmg_") else pd.NA,
            "none": pd.NA,
        }

        df[coluna] = pd.to_numeric(
            df[coluna].replace(substituicoes),
            errors="raise"
        )

    # Define tipos
    colunas_inteiras = [
        "age", "pop", "range", "hp",
        "armor_hack", "armor_pierce", "armor_crush",
        "los", "train_time",
    ]

    colunas_decimais = [
        "dmg_hack", "dmg_pierce", "dmg_crush", "speed",
    ]

    df[colunas_inteiras] = df[colunas_inteiras].astype("Int64")
    df[colunas_decimais] = df[colunas_decimais].astype("Float64")

    # Remove duplicatas
    df = df.drop_duplicates().reset_index(drop=True)

    # Valida valores
    for coluna in ["armor_hack", "armor_pierce", "armor_crush"]:
        if (df[coluna].dropna() > 100).any():
            raise ValueError(f"Armadura inválida: {coluna}")

    # Organiza colunas
    ordem = [
        "unit", "unit_class", "age",
        "cost_food", "cost_wood", "cost_gold", "cost_favor",
        "pop", "hp",
        "dmg_hack", "dmg_pierce", "dmg_crush",
        "armor_hack", "armor_pierce", "armor_crush",
        "range", "speed", "los", "train_time",
        "bonus", "other",
    ]

    return df[ordem]


In [13]:
df_limpo = limpar_dataset(df)

In [14]:
display(df_limpo)

,unit,unit_class,age,cost_food,cost_wood,cost_gold,cost_favor,pop,hp,dmg_hack,dmg_pierce,dmg_crush,armor_hack,armor_pierce,armor_crush,range,speed,los,train_time,bonus,other
0,Pegasus,Myth Unit,1,50,0,0,2,2,140,0.0,0.0,0.0,50,50,99,0,5.0,18,6,NaN,NaN
1,Centaur,Myth Unit,2,0,200,0,15,3,220,0.0,12.0,0.0,25,35,80,12,5.0,20,19,"Myth units, -Heroes",NaN
2,Cyclops,Myth Unit,2,250,0,0,22,4,600,15.0,0.0,12.0,40,50,80,2,3.2,16,20,Myth units,NaN
3,Minotaur,Myth Unit,2,200,0,0,16,4,300,15.0,0.0,10.0,60,50,80,2,4.0,20,20,Myth units,NaN
4,Hydra,Myth Unit,3,250,0,0,28,5,800,20.0,0.0,10.0,60,40,80,3,4.0,16,20,"Myth units, -Heroes",NaN
5,Manticore,Myth Unit,3,0,300,0,28,5,420,0.0,11.0,0.0,30,60,80,16,4.3,20,17,"Myth units, -Heroes",NaN
6,Nemean Lion,Myth Unit,3,0,0,200,25,4,660,20.0,0.0,10.0,30,60,80,1,4.8,16,20,"Myth units, -Heroes",NaN
7,Colossus,Myth Unit,4,0,0,300,40,5,1100,20.0,0.0,50.0,50,80,80,3,2.4,16,24,"Myth units, -Heroes",NaN
8,Chimera,Myth Unit,4,0,0,300,30,4,800,20.0,0.0,0.0,60,60,80,1,5.3,16,20,"Myth units, -Heroes",NaN
9,Medusa,Myth Unit,4,0,0,250,40,5,360,0.0,15.0,12.5,60,70,80,10,4.3,18,20,"Myth units, -Heroes",NaN


Descobrir quantos dados tem de cada classe e quais coluna tem valor ausente

In [15]:
print(df_limpo.isna().sum()[lambda s: s > 0])   #colunas com valores ausentes
df_limpo["unit_class"].value_counts()

age            2
pop            1
range          1
train_time     3
bonus         17
other         73
dtype: int64


unit_class
Myth Unit          30
Hero               18
Animal              7
Siege               6
Cavalry             6
Melee Infantry      5
Ship                5
Ranged Infantry     4
Spear Infantry      1
Ranged Cavalry      1
Name: count, dtype: int64

Carregar o conjunto sem classe e escolher atributos

In [18]:
caminho_noClass = "https://www-usr.inf.ufsm.br/~joaquim/UFSM/DM/ds/AgeOfMythology_noClass.csv"
teste = pd.read_csv(caminho, dtype="string", keep_default_na=False, encoding="utf-8-sig")

In [19]:
teste

,Unit,Age,Cost,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,LOS,Train time,Bonus,Other,Class
0,Pegasus,1,50F 2Fv,2,--,--,--,0,140,50,50,99,5,18,6,,,Myth Unit
1,Centaur,2,200W 15Fv,3,--,12,--,12,220,25,35,80,5,20,19,"Myth units, -Heroes",,Myth Unit
2,Cyclops,2,250F 22Fv,4,15,--,12,2,600,40,50,80,3.2,16,20,Myth units,,Myth Unit
3,Minotaur,2,200F 16Fv,4,15,--,10,2,300,60,50,80,4,20,20,Myth units,,Myth Unit
4,Hydra,3,250F 28Fv,5,20,--,10,3,800,60,40,80,4,16,20,"Myth units, -Heroes",,Myth Unit
5,Manticore,3,300W 28Fv,5,--,11**,--,16,420,30,60,80,4.3,20,17,"Myth units, -Heroes",,Myth Unit
6,Nemean Lion,3,200G 25Fv,4,20,--,10,1,660,30,60,80,4.8,16,20,"Myth units, -Heroes",,Myth Unit
7,Colossus,4,300G 40Fv,5,20,--,50,3,1100_,50,80,80,2.4,16,24,"Myth units, -Heroes",,Myth Unit
8,Chimera,4,300G 30Fv,4,20,--,--,1,800,60,60,80,5.3,16,20,"Myth units, -Heroes",,Myth Unit
9,Medusa,4,250G 40Fv,5,--,15,12.5,10,360,60,70,80,4.3,18,20,"Myth units, -Heroes",,Myth Unit


In [ ]:
# padronizar 
mapa = {
    "Age": "age", "Pop": "pop",
    "DmgHack": "dmg_hack", "DmgPierce": "dmg_pierce", "DmgCrush": "dmg_crush",
    "Range": "range", "HP": "hp",
    "ArmorHack": "armor_hack", "ArmorPierce": "armor_pierce", "ArmorCrush": "armor_crush",
    "Speed": "speed", "Train time": "train_time", "Favor": "cost_favor",
}
teste = teste.rename(columns=mapa)

FEATURES = list(mapa.values())
treino = df_limpo[["unit_class"] + FEATURES].copy()
treino.head()

,unit_class,age,pop,dmg_hack,dmg_pierce,dmg_crush,range,hp,armor_hack,armor_pierce,armor_crush,speed,train_time,cost_favor
0,Myth Unit,1,2,0.0,0.0,0.0,0,140,50,50,99,5.0,6,2
1,Myth Unit,2,3,0.0,12.0,0.0,12,220,25,35,80,5.0,19,15
2,Myth Unit,2,4,15.0,0.0,12.0,2,600,40,50,80,3.2,20,22
3,Myth Unit,2,4,15.0,0.0,10.0,2,300,60,50,80,4.0,20,16
4,Myth Unit,3,5,20.0,0.0,10.0,3,800,60,40,80,4.0,20,28


In [ ]:

print(df_limpo.loc[df_limpo[FEATURES].isna().any(axis=1), ["unit", "unit_class"] + FEATURES])

                unit unit_class   age   pop  dmg_hack  dmg_pierce  dmg_crush  \
14               Roc  Myth Unit     3     3       0.0         0.0        0.0   
42           Pharaoh       Hero     1     0       0.0        12.0        0.0   
43     Son of Osiris       Hero  <NA>  <NA>      60.0         0.0        0.0   
45  Hero of Ragnarok       Hero  <NA>     3      11.0         0.0        0.0   

    range   hp  armor_hack  armor_pierce  armor_crush  speed  train_time  \
14   <NA>  350          30            25           80    4.8          14   
42      4  100          15            15           99    4.0        <NA>   
43     18  546          30            50           99    3.6        <NA>   
45      1  120          30            30           99    4.8        <NA>   

    cost_favor  
14          15  
42           0  
43           0  
45           0  


Ajustar o valores ausentes para treino 

In [24]:
treino[FEATURES] = treino[FEATURES].fillna(0).astype(float)
teste = teste[FEATURES].astype(float)

X = treino[FEATURES]
y = treino["unit_class"]

KeyError: "None of [Index(['age', 'pop', 'dmg_hack', 'dmg_pierce', 'dmg_crush', 'range', 'hp',\n       'armor_hack', 'armor_pierce', 'armor_crush', 'speed', 'train_time',\n       'cost_favor'],\n      dtype='str')] are in the [columns]"

In [ ]:
treino.groupby("unit_class")[["hp", "armor_crush", "cost_favor", "range", "speed"]].mean().round(1)